# 06 · Modelo Late Fusion

**Objetivo:** Entrenar dos modelos independientes (uno para imágenes, otro para tabular) y combinar sus predicciones mediante un **clasificador aprendido** (SVM con kernel RBF).

**Datos de entrada:** `../data/raw/hnmist_28_28_RGB.csv`, `../data/raw/HAM10000_metadata.csv`

**Resultado esperado:** Modelos guardados en `../models/late_fusion_image_model.h5` y `../models/late_fusion_tabular_model.h5` con ~75% de accuracy combinado.

**Por qué Late Fusion es inferior a Early Fusion:** cada rama aprende de forma aislada, sin poder capturar las interacciones entre imagen y metadatos. La fusión en predicciones es una combinación de opiniones independientes, no un aprendizaje conjunto.

**Diferencia con un promedio fijo:** en lugar de usar pesos arbitrarios (`0.6 * img + 0.4 * tab`), entrenamos un SVM que **aprende** cuánto peso dar a cada modelo según los datos.

## Carga de datos, preprocesado y entrenamiento

**Estrategia de fusión:** las probabilidades de ambos modelos se concatenan (14 valores: 7 del modelo de imagen + 7 del tabular) y se usan como entrada de un SVM con kernel RBF. El SVM aprende la combinación óptima durante el entrenamiento.

```
pred_img  (n, 7)  ──┐
                     ├─→ concat (n, 14) ─→ SVM ─→ diagnóstico final
pred_tab  (n, 7)  ──┘
```

In [ ]:
#Por último, vamos a hacer un modelo Late-fusion. En este tipo de modelos, se fusionan datos de distintas categorías 
#Al FINAL del proceso, 
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Concatenate, BatchNormalization
import matplotlib.pyplot as plt
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
#Cargamos CSV 
# ── Carga y alineamiento por image_id ────────────────────────────────
df_images = pd.read_csv(r"../data/raw/hnmist_28_28_RGB.csv")
metadata  = pd.read_csv(r"../data/raw/HAM10000_metadata.csv")

n = min(len(df_images), len(metadata))
df_images = df_images.iloc[:n].reset_index(drop=True)
metadata  = metadata.iloc[:n].reset_index(drop=True)
df_images['image_id'] = metadata['image_id']

df = df_images.merge(
    metadata[['image_id', 'dx', 'age', 'sex', 'localization']],
    on='image_id'
)

le = LabelEncoder()
y_encoded = le.fit_transform(df['dx'].values)
y_onehot  = to_categorical(y_encoded)
num_classes = y_onehot.shape[1]

pixel_cols = [c for c in df.columns if c not in ['image_id', 'dx', 'age', 'sex', 'localization']]
X_img = df[pixel_cols].values.astype(np.float32).reshape(-1, 28, 28, 3)
X_img /= 255.0

tabular_data = pd.get_dummies(df[['age', 'sex', 'localization']], columns=['sex', 'localization'], drop_first=True)
X_tab = tabular_data.values.astype(np.float32)

# ── Split 70 / 15 / 15 ────────────────────────────────────────────────
X_img_train, X_img_temp, X_tab_train, X_tab_temp, y_train, y_temp, enc_train, enc_temp = train_test_split(
    X_img, X_tab, y_onehot, y_encoded,
    test_size=0.30, random_state=42, stratify=y_encoded)
X_img_val, X_img_test, X_tab_val, X_tab_test, y_val, y_test = train_test_split(
    X_img_temp, X_tab_temp, y_temp,
    test_size=0.50, random_state=42, stratify=enc_temp)

print(f"Train: {X_img_train.shape[0]} | Val: {X_img_val.shape[0]} | Test: {X_img_test.shape[0]}")

# ── Imputación: fit solo en train ─────────────────────────────────────
from sklearn.impute import SimpleImputer
X_tab_train = np.where(np.isinf(X_tab_train), np.nan, X_tab_train)
X_tab_val   = np.where(np.isinf(X_tab_val),   np.nan, X_tab_val)
X_tab_test  = np.where(np.isinf(X_tab_test),  np.nan, X_tab_test)

imp = SimpleImputer(strategy='median')
X_tab_train = imp.fit_transform(X_tab_train)
X_tab_val   = imp.transform(X_tab_val)
X_tab_test  = imp.transform(X_tab_test)

#Modelo tabular
input_tab = Input(shape=(X_tab_train.shape[1],))

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

model_tab = Sequential([
    Dense(32, activation='relu', input_shape=(X_tab_train.shape[1],)),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(num_classes, activation='softmax')
])
        # Compilar 
model_tab.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
model_tab.summary()
y_train = y_train.astype('float32')
y_val   = y_val.astype('float32')
y_test  = y_test.astype('float32')
        # Entrenamiento
early_stop = EarlyStopping(
    monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True
)

history_tab = model_tab.fit(
    X_tab_train, y_train,
    validation_data=(X_tab_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

#Evaluación 
import matplotlib.pyplot as plt

# Gráfica de pérdida
plt.figure(figsize=(8, 5))
plt.plot(history_tab.history['loss'], label='Training loss')
plt.plot(history_tab.history['val_loss'], label='Validation loss')
plt.title('Evolución de la pérdida (loss) durante el entrenamiento')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Modelo de imágenes
input_shape = (28, 28, 3)
num_classes = y_train.shape[1]

model_img = Sequential()
model_img.add(Conv2D(32, kernel_size = (3,3), activation='relu', padding='same', input_shape=input_shape, strides = (1,1)))
model_img.add(BatchNormalization())
model_img.add(MaxPooling2D(pool_size=(2,2)))
model_img.add(Conv2D(64, kernel_size = (3,3), activation='relu', padding='same', input_shape=input_shape, strides = (1,1)))
model_img.add(BatchNormalization())
model_img.add(MaxPooling2D(pool_size=(2,2)))
model_img.add(Conv2D(128, kernel_size = (3,3), activation='relu', padding='same', input_shape=input_shape, strides = (1,1)))
model_img.add(BatchNormalization())
model_img.add(MaxPooling2D(pool_size=(2,2)))
model_img.add(Conv2D(256, kernel_size = (3,3), activation='relu', padding='same', input_shape=input_shape, strides = (1,1)))
model_img.add(BatchNormalization())
model_img.add(MaxPooling2D(pool_size=(2,2)))
model_img.add(GlobalAveragePooling2D())
model_img.add(Dense(256, activation='relu'))
model_img.add(Dropout(0.5))
model_img.add(Dense(num_classes, activation='softmax'))

        # Compilar 
model_img.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True)
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow(X_img_train, y_train, batch_size=64)

history_img = model_img.fit(
    train_gen,
    validation_data=(X_img_val, y_val),
    epochs=50,
    callbacks=[early_stop]
)
#Evaluación del modelo 

# Obtenemos probabilidades sobre TRAIN+VAL para entrenar el SVM
# (el SVM es un modelo separado; val ya no se usa para early stopping)
pred_img_train = model_img.predict(np.concatenate([X_img_train, X_img_val]))
pred_tab_train = model_tab.predict(np.concatenate([X_tab_train, X_tab_val]))
y_train_svm    = np.argmax(np.concatenate([y_train, y_val]), axis=1)

# Obtenemos probabilidades sobre TEST (para evaluar)
pred_img_test = model_img.predict(X_img_test)
pred_tab_test = model_tab.predict(X_tab_test)

import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Concatenamos las probabilidades: (n, 7) + (n, 7) = (n, 14)
fusion_train = np.concatenate([pred_img_train, pred_tab_train], axis=1)
fusion_test  = np.concatenate([pred_img_test,  pred_tab_test],  axis=1)

y_true = np.argmax(y_test, axis=1)

# Entrenamos un SVM que aprende la combinación óptima entre ambos modelos
print("Entrenando SVM de fusión...")
clf = SVC(kernel='rbf', probability=True, random_state=42)
clf.fit(fusion_train, y_train_svm)

# Evaluamos
y_pred_classes = clf.predict(fusion_test)
acc = accuracy_score(y_true, y_pred_classes)
print(f"Accuracy Late Fusion (SVM): {acc:.4f}")

# guardamos el modelo late fusion
model_img.save('../models/late_fusion_image_model.h5')
model_tab.save('../models/late_fusion_tabular_model.h5')
# Graficamos resultados del modelo de imágenes

import matplotlib.pyplot as plt

plt.plot(history_img.history['accuracy'], label='Train Accuracy')
plt.plot(history_img.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Evaluación detallada por clase

El accuracy global no es suficiente en diagnóstico médico. Aquí vemos cuánto acierta el modelo en **cada tipo de lesión** por separado, y lo comparamos con el **baseline ZeroR** (lo que conseguiría un modelo que predice siempre la clase más frecuente, sin aprender nada).

In [ ]:
from sklearn.metrics import classification_report
from collections import Counter

# Baseline ZeroR
most_common = Counter(y_true).most_common(1)[0][1]
baseline = most_common / len(y_true)
print(f'Baseline ZeroR (predecir siempre la clase más frecuente): {baseline:.2%}')
print(f'Accuracy Late Fusion (SVM): {acc:.2%}')
print(f'Mejora sobre baseline: {acc - baseline:+.2%}')

print('
Resultados por clase diagnóstica:')
print(classification_report(y_true, y_pred_classes, target_names=le.classes_))

# Modelo Late fusion: Conclusiones
En este modelo hemos alcanzado un valor de 0.75 aprox. el cual no es malo, pero es peor que en nuestro modelo Early fusion. 
Vemos que esto sucede porque, al procesar las dos modalidades de forma independiente hasta el final, la red no puede aprender interacciones profundas entre modalidades (es decir, yo he configurado que el peso de la parte tabular sea 0.4, y el peso de la parte de imágenes sea 0.6, pero no es una manera muy compleja de hacerlo)
Además, el aprendizaje para clasificar ambas modalidades no se hace de manera conjunta. 
Este tipo de red se utiliza en casos donde se busca la modularidad y la robustez, dado que al no tener interacciones tan complejas entre las dimensiones, es fácil sustituir una variable por otra
